In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.window import Window
import pandas as pd

#LOAD info
spark = SparkSession.builder.appName('F1 Pipeline Notebook')\
    .master('local[*]')\
    .getOrCreate()

# Column name + typing
schema = T.StructType([
    T.StructField('Driver', T.StringType(), True),                   # Nom du pilote
    T.StructField('LapNumber', T.IntegerType(), True),               # Numéro du tour
    T.StructField('Compound', T.StringType(), True),                 # Type de pneu (Soft, Medium, Hard)
    T.StructField('Stint', T.IntegerType(), True),                   # Numéro de stint (séquence de tours avec le même pneu)
    T.StructField('TyreLife', T.DoubleType(), True),                 # Durée de vie du pneu
    T.StructField('Position', T.IntegerType(), True),                # Position du pilote
    T.StructField('LapTime (s)', T.DoubleType(), True),              # Temps du tour (en secondes)
    T.StructField('Race', T.StringType(), True),                     # Nom de la course
    T.StructField('Year', T.IntegerType(), True),                    # Année de la course
    T.StructField('LapTime_Delta', T.DoubleType(), True),            # Différence de temps par rapport au pilote devant
    T.StructField('Cumulative_Degradation', T.DoubleType(), True),   # Dégradation cumulative du pneu
    T.StructField('PitStop', T.IntegerType(), True),                 # Indicateur de pit stop (1 si le pilote a effectué un pit stop à ce tour, 0 sinon)
    T.StructField('PitNextLap', T.IntegerType(), True),              # Indicateur de pit stop au tour suivant (1 si le pilote effectuera un pit stop au tour suivant, 0 sinon)
    T.StructField('RaceProgress', T.DoubleType(), True),             # Progression de la course (en pourcentage)
    T.StructField('Normalized_TyreLife', T.DoubleType(), True),      # Durée de vie du pneu normalisée par rapport à la durée de vie maximale observée pour ce type de pneu
    T.StructField('Position_Change', T.DoubleType(), True),          # Changement de position par rapport au tour précédent (en nombre de places gagnées ou perdues)
])

csv_path = 'f1_strategy_dataset_v4.csv'
df = spark.read.csv(csv_path, header=True, schema=schema)\
    .withColumnRenamed('LapTime (s)', 'LapTime_s')



+------+---------+--------+-----+--------+--------+---------+--------------------+----+-------------------+----------------------+-------+----------+------------------+-------------------+---------------+
|Driver|LapNumber|Compound|Stint|TyreLife|Position|LapTime_s|                Race|Year|      LapTime_Delta|Cumulative_Degradation|PitStop|PitNextLap|      RaceProgress|Normalized_TyreLife|Position_Change|
+------+---------+--------+-----+--------+--------+---------+--------------------+----+-------------------+----------------------+-------+----------+------------------+-------------------+---------------+
|   ALB|        1|  MEDIUM|    1|     2.0|      17|  100.625|Abu Dhabi Grand Prix|2023|                0.0|                   0.0|      0|         0|0.0172413793103448| 0.1176470588235294|            0.0|
|   ALB|        2|  MEDIUM|    1|     3.0|      18|    93.56|Abu Dhabi Grand Prix|2023| -7.064999999999998|    -7.064999999999998|      0|         0|0.0344827586206896| 0.176470588

In [33]:
# header rapide 
# utilisé des drivers distincts
driver_filter = [row['Driver'] for row in df.select('Driver').distinct().collect()][:5]
print("Drivers présents dans le dataset :", driver_filter)
# on partitionne par driver et on ordonne par lap number decroissant pour avoir le dernier tour de chaque driver
w = Window.partitionBy('Driver').orderBy(F.col('LapNumber').desc())
one_row_per_driver = (
    df.filter(F.col('Driver').isin(driver_filter))
      .withColumn('rn', F.row_number().over(w))
      .filter(F.col('rn') == 1)
      .drop('rn')
)
one_row_per_driver.show(len(driver_filter), truncate=False)




Drivers présents dans le dataset : ['OCO', 'BOT', 'HAM', 'VER', 'ZHO']
+------+---------+------------+-----+--------+--------+---------+-----------------+----+-------------------+----------------------+-------+----------+------------+-------------------+---------------+
|Driver|LapNumber|Compound    |Stint|TyreLife|Position|LapTime_s|Race             |Year|LapTime_Delta      |Cumulative_Degradation|PitStop|PitNextLap|RaceProgress|Normalized_TyreLife|Position_Change|
+------+---------+------------+-----+--------+--------+---------+-----------------+----+-------------------+----------------------+-------+----------+------------+-------------------+---------------+
|BOT   |77       |INTERMEDIATE|2    |26.0    |11      |84.139   |Monaco Grand Prix|2023|-0.4980000000000046|-8.015                |0      |0         |1.0         |1.0                |0.0            |
|HAM   |78       |INTERMEDIATE|3    |24.0    |4       |85.105   |Monaco Grand Prix|2023|-1.1049999999999898|-14.49799999999999   

In [64]:
# Transformation 

# Cette partie nous permet d'ajouter des informations de performances
# par rapport au tours précédents pour chaque pilote dans chaque course,
# ainsi que des informations sur les changements de position.

# Utilisation de Window :
# - Window.partitionBy('Race', 'Driver') : partitionne les lignes par course et par pilote,
#   permettant de faire des calculs indépendants pour chaque pilote dans chaque course.
#   Similaire a un GroupBy 
# - w.rowsBetween(Window.unboundedPreceding, 0) : fenêtre allant du premier enregistrement de la partition
#   jusqu'à la ligne courante 
# Dans notre cas prend du debut (unboundedPreceding) jusqu'à la ligne courante (index[0])
# pour calculer la dégradation cumulative.

w = Window.partitionBy('Race', 'Driver').orderBy('LapNumber')

df.show(5)

df = df.withColumn('prev_LapTime', F.lag('LapTime_s').over(w))\
    .withColumn('LapTime_Delta_recalc',
                F.coalesce(F.col('LapTime_s') - F.col('prev_LapTime'), F.lit(0.0)))\
    .withColumn('Cumulative_Degradation_recalc',
                F.sum('LapTime_Delta_recalc')
                    .over(w.rowsBetween(Window.unboundedPreceding, 0)))\
    .withColumn('prev_Position', F.lag('Position').over(w))\
    .withColumn('Position_Change_recalc',
                F.coalesce(F.col('prev_Position') - F.col('Position'), F.lit(0.0)))

print("------------------------\nExemple de calculs de performance par rapport au tour précédent :")
df.show(5, truncate=False)

# 1) prev_LapTime : récupère le laptime du tour précédent pour le même pilote/ course (None si pas de précédent).
# 2) LapTime_Delta_recalc : différence entre le laptime courant et le laptime précédent.
#    Si prev_LapTime est null (premier tour), on met 0.0 pour éviter les valeurs nulles dans les calculs suivants.
# 3) Cumulative_Degradation_recalc : somme cumulée de 'LapTime_Delta_recalc' depuis le début de la partition
#    (i.e. dégradation cumulative du temps au fil des tours pour ce pilote dans cette course).
# 4) prev_Position : position du tour précédent pour le même pilote/ course (None si pas de précédent).
# 5) Position_Change_recalc : variation de position entre le tour précédent et le tour courant.
#    Si prev_Position est null, on met 0.0 pour signifier pas de changement au premier enregistrement.



+------+---------+--------+-----+--------+--------+---------+--------------------+----+-------------------+----------------------+-------+----------+------------------+-------------------+---------------+
|Driver|LapNumber|Compound|Stint|TyreLife|Position|LapTime_s|                Race|Year|      LapTime_Delta|Cumulative_Degradation|PitStop|PitNextLap|      RaceProgress|Normalized_TyreLife|Position_Change|
+------+---------+--------+-----+--------+--------+---------+--------------------+----+-------------------+----------------------+-------+----------+------------------+-------------------+---------------+
|   ALB|        1|  MEDIUM|    1|     2.0|      17|  100.625|Abu Dhabi Grand Prix|2023|                0.0|                   0.0|      0|         0|0.0172413793103448| 0.1176470588235294|            0.0|
|   ALB|        2|  MEDIUM|    1|     3.0|      18|    93.56|Abu Dhabi Grand Prix|2023| -7.064999999999998|    -7.064999999999998|      0|         0|0.0344827586206896| 0.176470588

In [75]:
# SHOW INFO

# Affiche les meilleurs tours par pilote (meilleur = temps minimal).
# On récupère tous les pilotes triés par meilleur tour
# on convertit en pandas et on affiche tout dans le terminal. (meilleur affichage)
print('-----------------------\nMeilleurs tours par pilote (tous les pilotes, triés par BestLap) :')
best_laps = df.groupBy('Driver')\
    .agg(F.min('LapTime_s').alias('BestLap'))\
    .orderBy('BestLap')
    
# pandas pour lisibilité 
best_laps_pdf = best_laps.toPandas()
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)
best_laps_pdf['BestLap'] = best_laps_pdf['BestLap'].round(3)
print(best_laps_pdf.to_string(index=False))



-----------------------
Meilleurs tours par pilote (tous les pilotes, triés par BestLap) :
Driver  BestLap
   VER   67.012
   LEC   67.583
   ALO   67.694
   PIA   67.924
   NOR   68.016
   PER   68.111
   STR   68.463
   HAM   68.628
   SAI   68.880
   GAS   69.046
   RUS   69.075
   BOR   69.247
   HUL   69.459
   OCO   69.550
   ALB   69.560
   SAR   69.611
   TSU   69.620
   COL   69.621
   ZHO   69.786
   DEV   69.852
   BOT   69.940
   BEA   69.960
   LAW   69.977
   MAG   70.125
   HAD   70.204
   RIC   70.426
   VET   70.467
   MSC   70.479
   LAT   70.890
   ANT   73.123
   DOO   89.121
